# Dataset

In [ ]:
import torch
from src.options import opt_dict
from src.data.realcamvid_dataset import RealcamvidDataset

opt = opt_dict["wan2.1_t2v_1.3b"]
dataset = RealcamvidDataset(opt, training=True)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=2, shuffle=True)

In [ ]:
data = next(iter(dataloader))
for k, v in data.items():
    print(f"{k}: {v.shape if isinstance(v, torch.Tensor) else v}")

import imageio.v2 as iio
from src.utils import *

iio.mimwrite("temp.mp4", tensor_to_video(data["image"]))
print(data["C2W"][0, 0], data["fxfycxcy"][0, 0])

# Visualize DA3

In [ ]:
import imageio.v2 as iio
import numpy as np
from decord import VideoReader, cpu
import torchvision.transforms as tvT
from src.options import ROOT
from src.utils import *

x = np.load(f"{ROOT}/data/RealCam-Vid-DA3/RealEstate10K/train/mHnV_xALVho/6ae12d32b5ba6faa.npz")
vr = VideoReader(str(f"{ROOT}/data/RealCam-Vid/RealEstate10K/train/mHnV_xALVho/6ae12d32b5ba6faa.mp4"), ctx=cpu(0))
v = vr[:].asnumpy()

depths, confs, W2C, intrinsics = x["depth"], x["conf"], x["extrinsics"], x["intrinsics"]
print(depths.shape, confs.shape, W2C.shape, intrinsics.shape)

v = torch.from_numpy(v).permute(0, 3, 1, 2) / 255.
v = tvT.Resize((280, 504), tvT.InterpolationMode.BICUBIC)(v).clamp(0., 1.)
print(v.shape)


In [ ]:
depths = torch.from_numpy(depths).float()
confs = torch.from_numpy(confs).float()

W2C_ = torch.eye(4).unsqueeze(0).repeat(W2C.shape[0], 1, 1)
W2C_[:, :3, :4] = torch.from_numpy(W2C).float()
C2W = inverse_c2w(W2C_)
intrinsics[:, 0, 0] /= 504
intrinsics[:, 1, 1] /= 280
intrinsics[:, 0, 2] /= 504
intrinsics[:, 1, 2] /= 280
fxfycxcy = intrinsics_to_fxfycxcy(torch.from_numpy(intrinsics).float()[None, ...])[0]

In [ ]:
iio.mimwrite("temp.mp4", tensor_to_video(v))
iio.mimwrite("temp_depth.mp4", tensor_to_video(colorize_depth(depths)))
iio.mimwrite("temp_conf.mp4", tensor_to_video(colorize_depth(confs)))

xyzs = unproject_depth(depths[None, ...], C2W[None, ...], fxfycxcy[None, ...])[0]
xyzs = normalize_among_last_dims(rearrange(xyzs, "f c h w -> c f h w"), num_dims=3)
xyzs = rearrange(xyzs, "c f h w -> f c h w")
iio.mimwrite("temp_xyzs.mp4", tensor_to_video(xyzs))

save_xyz_rgb_as_ply("temp.ply", xyzs, v, ratio=0.1)